<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2013%20-%202026-05-11%20-%20Ensembles%20y%20Random%20Forest/clase_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 13 · Ensembles y Random Forest

**Seminario EDA 2026 — Semana 4, Día 1 (lunes 11 de mayo)**

> Pasamos del árbol único — alta varianza — al **bosque**: muchos árboles trabajando juntos. Aprendemos:
>
> 1. **Bagging** (Bootstrap Aggregating) y por qué reduce la varianza.
> 2. **Random Forest** = bagging + selección aleatoria de features en cada split.
> 3. Score **OOB** (Out-Of-Bag) — validación gratis.
> 4. Diferencia conceptual con **Boosting** (Gradient Boosting).

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, BaggingClassifier,
                              GradientBoostingClassifier)
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, classification_report

SEED = 42
np.random.seed(SEED)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
print('Setup OK')

## 1. Mismo dataset que Días 10–12 (Pima Diabetes) para comparar

In [ ]:
URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
        'Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']
df = pd.read_csv(URL, names=cols)
X = df.drop(columns='Outcome')
y = df['Outcome'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)

## 2. Recordatorio: el árbol único tenía alta varianza

Para verlo en vivo, entrenamos 5 árboles sobre 5 muestras bootstrap del train y comparamos sus AUC en test.

In [ ]:
rng = np.random.default_rng(SEED)
n = len(X_train)
aucs_individuales = []
for i in range(5):
    idx = rng.integers(0, n, size=n)            # muestra bootstrap (con reemplazo)
    Xb = X_train.iloc[idx]; yb = y_train[idx]
    t = DecisionTreeClassifier(random_state=SEED+i).fit(Xb, yb)
    auc = roc_auc_score(y_test, t.predict_proba(X_test)[:, 1])
    aucs_individuales.append(auc)
    print(f'Árbol {i+1}: AUC test = {auc:.3f}')

print(f'\nMedia AUC: {np.mean(aucs_individuales):.3f}  ·  Std: {np.std(aucs_individuales):.3f}')

**La varianza entre árboles individuales es notoria** (típicamente AUC oscila entre 0.65 y 0.78). Esa varianza es justamente lo que el bagging reduce.

## 3. Bagging — 100 árboles cada uno con su muestra bootstrap

scikit-learn lo implementa en `BaggingClassifier`. Predicción final = voto mayoritario (clasificación) o promedio (regresión).

In [ ]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    n_estimators=100,
    bootstrap=True,
    oob_score=True,
    n_jobs=-1,
    random_state=SEED
).fit(X_train, y_train)

auc_bag = roc_auc_score(y_test, bag.predict_proba(X_test)[:, 1])
print(f'AUC test (bagging 100 árboles): {auc_bag:.3f}')
print(f'OOB score                    : {bag.oob_score_:.3f}')

**¿Por qué OOB es validación gratis?**  
Cada muestra bootstrap deja afuera ~37% de los datos (el `1 - (1-1/n)^n → 1/e ≈ 0.368`). Esos quedan disponibles para evaluar cada árbol sin tener que reservar un test set. En datasets pequeños es oro.

## 4. Random Forest = Bagging + selección aleatoria de features

El truco extra: en cada split, el árbol solo "ve" un subconjunto aleatorio de features (`max_features`, default `sqrt(p)`). Esto **descorrelaciona** los árboles entre sí — y reducir la correlación reduce aún más la varianza promedio.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    max_features='sqrt',
    oob_score=True,
    n_jobs=-1,
    random_state=SEED
).fit(X_train, y_train)

auc_rf = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
print(f'AUC test (RF 300 árboles): {auc_rf:.3f}')
print(f'OOB score                : {rf.oob_score_:.3f}')
print(f'Acc test                 : {rf.score(X_test, y_test):.3f}')

## 5. ¿Cuántos árboles son suficientes?

Curva de AUC vs n_estimators. Spoiler: el AUC se estabiliza muy rápido (~100-200).

In [ ]:
ns = [5, 10, 25, 50, 100, 200, 400]
aucs = []
for n_est in ns:
    m = RandomForestClassifier(n_estimators=n_est, n_jobs=-1, random_state=SEED).fit(X_train, y_train)
    aucs.append(roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]))

plt.plot(ns, aucs, marker='o')
plt.xscale('log'); plt.xlabel('n_estimators'); plt.ylabel('AUC test')
plt.title('AUC vs número de árboles en RF')
plt.tight_layout(); plt.show()

## 6. Feature importance del bosque

RF promedia la importancia de cada feature sobre todos los árboles → suele ser más estable que la del árbol único.

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
imp.plot(kind='barh', figsize=(8, 5), color='#2980b9')
plt.title('Random Forest · feature importance')
plt.xlabel('Importancia (media sobre 300 árboles)')
plt.tight_layout(); plt.show()
print(imp.sort_values(ascending=False).round(3))

## 7. Boosting — la otra rama de los ensembles

**Bagging** entrena árboles en paralelo, todos del mismo "poder". **Boosting** entrena árboles secuencialmente, cada nuevo árbol aprende de los errores del anterior. `GradientBoostingClassifier` es el clásico; XGBoost / LightGBM son su versión moderna optimizada.

In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3,
                                random_state=SEED).fit(X_train, y_train)
auc_gb = roc_auc_score(y_test, gb.predict_proba(X_test)[:, 1])
print(f'AUC test (Gradient Boosting): {auc_gb:.3f}')

## 8. Tabla comparativa final — Pima Diabetes

Sentamos los 5 modelos lado a lado en CV-5.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

modelos = {
    'Logística':           Pipeline([('s', StandardScaler()),
                                     ('m', LogisticRegression(max_iter=1000, random_state=SEED))]),
    'Árbol (depth=4)':     DecisionTreeClassifier(max_depth=4, random_state=SEED),
    'Bagging (100)':       BaggingClassifier(n_estimators=100, n_jobs=-1, random_state=SEED),
    'Random Forest (300)': RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
    'Gradient Boost (200)':GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                      max_depth=3, random_state=SEED),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
filas = []
for nombre, m in modelos.items():
    aucs = cross_val_score(m, X, y, cv=skf, scoring='roc_auc')
    filas.append({'modelo': nombre, 'AUC_mean': aucs.mean(), 'AUC_std': aucs.std()})

tabla = pd.DataFrame(filas).sort_values('AUC_mean', ascending=False).round(4)
print(tabla.to_string(index=False))

**Lo que típicamente verás:**  
- Boosting y RF lideran (AUC ~0.83-0.84).  
- Logística sale sorprendentemente cerca cuando los datos son aprox. lineales.  
- Árbol único queda último (~0.75) — eso es lo que motivó los ensembles.

**Nota importante:** estamos sobre Pima (768 filas). En datasets más grandes los ensembles ganan por mucho más margen.

## 9. Ejercicio — sube `n_estimators` y observa OOB

In [ ]:
# TODO: cambia n_estimators a 50, 100, 200, 400 y reporta OOB para cada uno
for n_est in [50, 100, 200, 400]:
    rf_i = RandomForestClassifier(n_estimators=n_est, oob_score=True,
                                  n_jobs=-1, random_state=SEED).fit(X_train, y_train)
    print(f'n_estimators={n_est:>4d}   OOB={rf_i.oob_score_:.3f}')

In [ ]:
assert rf_i.oob_score_ > 0.5
print('✓ Ejercicio completado')

## 10. Entregable parcial S4 (avance)

Sobre el dataset de tu proyecto:

1. Entrena un `RandomForestClassifier` (o `Regressor`) con `n_estimators=300` y `oob_score=True`. Reporta OOB y AUC/R² en test.
2. Compara contra el árbol único de la S3. ¿Cuánto subió el AUC? ¿Bajó la varianza entre folds?
3. Reporta `feature_importances_` y compara con el ranking del árbol único — ¿cambió mucho?
4. *Opcional:* prueba `GradientBoostingClassifier` con `learning_rate ∈ {0.01, 0.05, 0.1}`.

Mañana (martes 12 may) usaremos pruebas estadísticas para validar si las diferencias entre modelos son significativas.